In [1]:
from pyspark.sql import SparkSession, functions as F

warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("Global Superstore - Silver ETL (Modelo World Data)")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .enableHiveSupport()
    .getOrCreate()
)
print("Spark pronto:", spark.version)

Spark pronto: 3.4.1


In [3]:
from pathlib import Path

BRONZE_PATH = "hdfs://hdfs-nn:9000/projeto/bronze/global_superstore"
SILVER_PATH = "hdfs://hdfs-nn:9000/warehouse/silver.db/global_superstore_silver"
TABELA_HIVE = "silver.global_superstore_silver"

RAW_DIR = "hdfs://hdfs-nn:9000/projeto/raw/global_superstore"
RAW_CSV_HDFS = RAW_DIR + "/global_superstore_orders.csv"
RAW_EXCEL_HDFS = RAW_DIR + "/global_superstore_original.xlsx"
CSV_LOCAL = "/home/jovyan/global_superstore_orders_raw.csv"
HTML_PATH = "/home/jovyan/relatorio_global_superstore_silver_modelo_world_data.html"

conf = spark.sparkContext._jsc.hadoopConfiguration()
PathHDFS = spark._jvm.org.apache.hadoop.fs.Path
bronze_hadoop_path = PathHDFS(BRONZE_PATH)
hdfs = bronze_hadoop_path.getFileSystem(conf)
JA_BRONZE = hdfs.exists(PathHDFS(BRONZE_PATH + "/_delta_log"))

candidatos = sorted([
    p for p in Path("/home/jovyan").glob("*.xlsx")
    if "superstore" in p.name.lower()
])
EXCEL_LOCAL = candidatos[0] if candidatos else None

print("Bronze já existe em Delta:", JA_BRONZE)
print("Excel encontrado:", EXCEL_LOCAL if EXCEL_LOCAL else "nenhum")
print("Bronze:", BRONZE_PATH)
print("Silver:", SILVER_PATH)

if not JA_BRONZE:
    assert EXCEL_LOCAL is not None, (
        "Falta o Excel original. Faz Upload de 'Global Superstore Data(1).xlsx' "
        "para /home/jovyan e executa esta célula novamente."
    )

Bronze já existe em Delta: False
Excel encontrado: /home/jovyan/Global Superstore Data.xlsx
Bronze: hdfs://hdfs-nn:9000/projeto/bronze/global_superstore
Silver: hdfs://hdfs-nn:9000/warehouse/silver.db/global_superstore_silver


In [4]:
if JA_BRONZE:
    print("Bronze já existe: não é necessário voltar a importar o Excel.")
else:
    import pandas as pd

    orders_pd = pd.read_excel(
        EXCEL_LOCAL,
        sheet_name="Orders",
        dtype=str,
        keep_default_na=False,
        engine="openpyxl"
    )
    print("Linhas da folha Orders:", len(orders_pd))
    print("Colunas da folha Orders:", len(orders_pd.columns))
    assert len(orders_pd) == 51290, "A folha Orders deveria ter 51 290 registos."
    assert len(orders_pd.columns) == 24, "A folha Orders deveria ter 24 colunas."

    # CSV intermédio: manter todos os atributos para a Bronze.
    orders_pd.to_csv(CSV_LOCAL, index=False, encoding="utf-8")

    pasta_hdfs = PathHDFS(RAW_DIR)
    hdfs.mkdirs(pasta_hdfs)

    hdfs.copyFromLocalFile(
        False, True, PathHDFS(str(EXCEL_LOCAL)), PathHDFS(RAW_EXCEL_HDFS)
    )
    hdfs.copyFromLocalFile(
        False, True, PathHDFS(CSV_LOCAL), PathHDFS(RAW_CSV_HDFS)
    )

    assert hdfs.exists(PathHDFS(RAW_CSV_HDFS)), "O CSV não foi copiado para o HDFS."
    assert hdfs.exists(PathHDFS(RAW_EXCEL_HDFS)), "O Excel não foi copiado para o HDFS."
    print("Excel original no HDFS:", RAW_EXCEL_HDFS)
    print("CSV da folha Orders no HDFS:", RAW_CSV_HDFS)

Linhas da folha Orders: 51290
Colunas da folha Orders: 24
Excel original no HDFS: hdfs://hdfs-nn:9000/projeto/raw/global_superstore/global_superstore_original.xlsx
CSV da folha Orders no HDFS: hdfs://hdfs-nn:9000/projeto/raw/global_superstore/global_superstore_orders.csv


In [5]:
import re

def to_snake(nome):
    nome = nome.replace("\ufeff", "").strip().lower()
    return re.sub(r"[^a-z0-9]+", "_", nome).strip("_")

if JA_BRONZE:
    bronze_df = spark.read.format("delta").load(BRONZE_PATH)
    print("Bronze existente carregada.")
else:
    raw_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(RAW_CSV_HDFS)
    )
    nomes = [to_snake(c) for c in raw_df.columns]
    assert len(nomes) == len(set(nomes)), "Colisões entre nomes de colunas normalizados."
    raw_df = raw_df.toDF(*nomes)
    print("Linhas RAW:", raw_df.count(), "| Colunas RAW:", len(raw_df.columns))
    assert raw_df.count() == 51290, "Quantidade inesperada de linhas na RAW."
    assert len(raw_df.columns) == 24, "Quantidade inesperada de colunas na RAW."

    (raw_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(BRONZE_PATH)
    )
    bronze_df = spark.read.format("delta").load(BRONZE_PATH)
    print("Bronze criada com sucesso em:", BRONZE_PATH)

if len(bronze_df.columns) == 24:
    bronze_df = bronze_df.toDF(*[to_snake(c) for c in bronze_df.columns])

assert bronze_df.count() == 51290, "Esperam-se 51 290 linhas na Bronze."
assert len(bronze_df.columns) == 24, "Esperam-se 24 colunas na Bronze."
print("Bronze:", bronze_df.count(), "registos e", len(bronze_df.columns), "colunas")
bronze_df.select("row_id", "order_date", "country", "sales", "profit").show(5, False)

Linhas RAW: 51290 | Colunas RAW: 24
Bronze criada com sucesso em: hdfs://hdfs-nn:9000/projeto/bronze/global_superstore
Bronze: 51290 registos e 24 colunas
+------+-------------------+-----------+------+------+
|row_id|order_date         |country    |sales |profit|
+------+-------------------+-----------+------+------+
|24599 |2017-03-22 00:00:00|Afghanistan|731.82|102.42|
|29465 |2015-09-01 00:00:00|Afghanistan|243.54|104.49|
|24598 |2017-03-22 00:00:00|Afghanistan|346.32|13.77 |
|24597 |2017-03-22 00:00:00|Afghanistan|169.68|79.68 |
|29464 |2015-09-01 00:00:00|Afghanistan|203.88|24.36 |
+------+-------------------+-----------+------+------+
only showing top 5 rows



In [6]:
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
bronze_df = bronze_df.toDF(*[to_snake(c) for c in bronze_df.columns])

necessarias = [
    "row_id", "order_date", "market", "country",
    "category", "sub_category", "sales", "discount", "profit"
]
faltam = [c for c in necessarias if c not in bronze_df.columns]
assert not faltam, f"Faltam colunas na Bronze: {faltam}"

print("Registos Bronze:", bronze_df.count())
print("Colunas Bronze:", len(bronze_df.columns))
bronze_df.select("row_id", "order_date", "country", "sales", "discount", "profit").show(5, False)

Registos Bronze: 51290
Colunas Bronze: 24
+------+-------------------+-----------+------+--------+------+
|row_id|order_date         |country    |sales |discount|profit|
+------+-------------------+-----------+------+--------+------+
|24599 |2017-03-22 00:00:00|Afghanistan|731.82|0       |102.42|
|29465 |2015-09-01 00:00:00|Afghanistan|243.54|0       |104.49|
|24598 |2017-03-22 00:00:00|Afghanistan|346.32|0       |13.77 |
|24597 |2017-03-22 00:00:00|Afghanistan|169.68|0       |79.68 |
|29464 |2015-09-01 00:00:00|Afghanistan|203.88|0       |24.36 |
+------+-------------------+-----------+------+--------+------+
only showing top 5 rows



In [7]:
# --- 2. TRANSFORM ---
n_inicial = bronze_df.count()
base = bronze_df.dropDuplicates(["row_id"])
duplicados_removidos = n_inicial - base.count()
print("Duplicados pelo Row ID removidos:", duplicados_removidos)

base = base.select(
    "order_date", "market", "country", "category", "sub_category",
    "sales", "discount", "profit"
)

# Texto sem espaços exteriores e vazio -> NULL.
for c in ["market", "country", "category", "sub_category"]:
    base = base.withColumn(c, F.trim(F.col(c).cast("string")))
    base = base.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))

# Datas do Excel exportadas para CSV; suportar vários formatos usuais.
data_texto = F.trim(F.col("order_date").cast("string"))
base = base.withColumn("order_date", F.coalesce(
    F.to_date(F.substring(data_texto, 1, 10), "yyyy-MM-dd"),
    F.to_date(F.substring(data_texto, 1, 10), "dd-MM-yyyy"),
    F.to_date(data_texto, "dd/MM/yyyy"),
    F.to_date(data_texto, "M/d/yyyy")
))
base = base.withColumn("ano", F.year("order_date"))

# Verificar se se perderam valores por conversão para número.
for c in ["sales", "discount", "profit"]:
    texto = F.trim(F.col(c).cast("string"))
    base = base.withColumn(c, F.when(
        texto.isNull() | (texto == ""),
        F.lit(None).cast("double")
    ).otherwise(texto.cast("double")))

# A Silver utiliza apenas linhas completas e coerentes nos campos essenciais.
antes_filtro = base.count()
base = base.filter(
    F.col("order_date").isNotNull() &
    F.col("market").isNotNull() &
    F.col("country").isNotNull() &
    F.col("category").isNotNull() &
    F.col("sub_category").isNotNull() &
    F.col("sales").isNotNull() & (F.col("sales") >= 0) &
    F.col("profit").isNotNull() &
    F.col("discount").isNotNull() & F.col("discount").between(0, 1)
)
linhas_invalidas = antes_filtro - base.count()
print("Linhas inválidas excluídas:", linhas_invalidas)

df = (
    base
    .withColumnRenamed("order_date", "data_encomenda")
    .withColumnRenamed("market", "mercado")
    .withColumnRenamed("country", "pais")
    .withColumnRenamed("category", "categoria")
    .withColumnRenamed("sub_category", "subcategoria")
    .withColumnRenamed("sales", "vendas")
    .withColumnRenamed("discount", "desconto")
    .withColumnRenamed("profit", "lucro")
)

# IQR: detetar valores extremos, sem os eliminar.
def limites_iqr(coluna, k=3.0):
    q1, q3 = df.approxQuantile(coluna, [0.25, 0.75], 0.001)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

v_lo, v_hi = limites_iqr("vendas")
l_lo, l_hi = limites_iqr("lucro")

df_final = (
    df
    .withColumn("flag_vendas_extremas", (F.col("vendas") < v_lo) | (F.col("vendas") > v_hi))
    .withColumn("flag_lucro_extremo", (F.col("lucro") < l_lo) | (F.col("lucro") > l_hi))
    .withColumn("flag_desconto_alto", F.col("desconto") >= 0.70)
    .withColumn("flag_prejuizo", F.col("lucro") < 0)
    .select(
        "data_encomenda", "ano", "mercado", "pais", "categoria", "subcategoria",
        "vendas", "desconto", "lucro", "flag_vendas_extremas",
        "flag_lucro_extremo", "flag_desconto_alto", "flag_prejuizo"
    )
)
df_final.cache()

print("Registos Silver:", df_final.count())
print("Colunas Silver:", len(df_final.columns))
print("Intervalo IQR vendas:", (round(v_lo, 2), round(v_hi, 2)))
print("Intervalo IQR lucro:", (round(l_lo, 2), round(l_hi, 2)))
df_final.printSchema()
df_final.select("data_encomenda", "pais", "vendas", "lucro").show(10, False)

Duplicados pelo Row ID removidos: 0
Linhas inválidas excluídas: 0
Registos Silver: 51290
Colunas Silver: 13
Intervalo IQR vendas: (-627.18, 907.92)
Intervalo IQR lucro: (-109.89, 146.52)
root
 |-- data_encomenda: date (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mercado: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- subcategoria: string (nullable = true)
 |-- vendas: double (nullable = true)
 |-- desconto: double (nullable = true)
 |-- lucro: double (nullable = true)
 |-- flag_vendas_extremas: boolean (nullable = true)
 |-- flag_lucro_extremo: boolean (nullable = true)
 |-- flag_desconto_alto: boolean (nullable = true)
 |-- flag_prejuizo: boolean (nullable = true)

+--------------+--------------+-------+-------+
|data_encomenda|pais          |vendas |lucro  |
+--------------+--------------+-------+-------+
|2016-12-04    |Brazil        |15.76  |-18.54 |
|2014-12-07    |Germany       |45.927 |18.837 |
|2014-08-30

In [8]:
# --- Verificações ---
n_bronze = bronze_df.count()
n_silver = df_final.count()
num_paises_nulos = df_final.filter(F.col("pais").isNull()).count()
num_datas_nulas = df_final.filter(F.col("data_encomenda").isNull()).count()
num_descontos_invalidos = df_final.filter(~F.col("desconto").between(0, 1)).count()

print("Bronze:", n_bronze, "| Silver:", n_silver)
print("Duplicados Row ID eliminados:", duplicados_removidos)
print("Linhas inválidas eliminadas:", linhas_invalidas)
print("Colunas Silver:", len(df_final.columns))
print("Países distintos:", df_final.select("pais").distinct().count())
print("Países nulos:", num_paises_nulos)
print("Datas nulas:", num_datas_nulas)
print("Descontos inválidos:", num_descontos_invalidos)

print("\nNulos por coluna:")
df_final.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_final.columns
]).show(truncate=False)

print("\nFlags:")
df_final.select([
    F.sum(F.col(c).cast("int")).alias(c)
    for c in df_final.columns if c.startswith("flag_")
]).show(truncate=False)

print("\nMargem de lucro e desconto médio (verificação anual):")
df_final.groupBy("ano").agg(
    F.round(F.sum("lucro") / F.sum("vendas") * 100, 2).alias("margem_lucro_pct"),
    F.round(F.avg("desconto") * 100, 2).alias("desconto_medio_pct")
).orderBy("ano").show()

assert n_bronze == 51290, "Bronze inesperada: verifica o dataset."
assert n_silver == 51290, "Perderam-se registos: investiga os filtros antes de gravar."
assert len(df_final.columns) == 13, "A Silver deve ter 13 colunas."
assert num_paises_nulos == 0, "Existem países sem nome."
assert num_datas_nulas == 0, "Existem datas inválidas."
assert num_descontos_invalidos == 0, "Existem descontos fora de [0,1]."
print("\nValidações concluídas. Pronto para gravar no HDFS.")

Bronze: 51290 | Silver: 51290
Duplicados Row ID eliminados: 0
Linhas inválidas eliminadas: 0
Colunas Silver: 13
Países distintos: 165
Países nulos: 0
Datas nulas: 0
Descontos inválidos: 0

Nulos por coluna:
+--------------+---+-------+----+---------+------------+------+--------+-----+--------------------+------------------+------------------+-------------+
|data_encomenda|ano|mercado|pais|categoria|subcategoria|vendas|desconto|lucro|flag_vendas_extremas|flag_lucro_extremo|flag_desconto_alto|flag_prejuizo|
+--------------+---+-------+----+---------+------------+------+--------+-----+--------------------+------------------+------------------+-------------+
|0             |0  |0      |0   |0        |0           |0     |0       |0    |0                   |0                 |0                 |0            |
+--------------+---+-------+----+---------+------------+------+--------+-----+--------------------+------------------+------------------+-------------+


Flags:
+--------------------+--

In [9]:
# --- 3. LOAD ---
spark.sql(
    f"CREATE DATABASE IF NOT EXISTS silver LOCATION '{warehouse_location}/silver.db'"
)

hadoop_path = PathHDFS(SILVER_PATH + "/_delta_log")
hadoop_fs = hadoop_path.getFileSystem(conf)
ja_delta = hadoop_fs.exists(hadoop_path)

if not ja_delta:
    print("Primeira execução: a guardar em Parquet...")
    (df_final.write
        .mode("overwrite")
        .format("parquet")
        .save(SILVER_PATH))

    # Registar no Hive como tabela EXTERNA em Parquet.
    spark.sql(f"DROP TABLE IF EXISTS {TABELA_HIVE}")
    spark.sql(f'''
        CREATE EXTERNAL TABLE {TABELA_HIVE} (
            data_encomenda DATE,
            ano INT,
            mercado STRING,
            pais STRING,
            categoria STRING,
            subcategoria STRING,
            vendas DOUBLE,
            desconto DOUBLE,
            lucro DOUBLE,
            flag_vendas_extremas BOOLEAN,
            flag_lucro_extremo BOOLEAN,
            flag_desconto_alto BOOLEAN,
            flag_prejuizo BOOLEAN
        )
        STORED AS PARQUET
        LOCATION '{SILVER_PATH}'
    ''')
    print("Parquet gravado e tabela externa Hive criada.")
else:
    print("O caminho já é Delta: a atualizar diretamente...")
    (df_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(SILVER_PATH))
    # Registar como Delta, se o Hive ainda tiver uma definição antiga.
    spark.sql(f"DROP TABLE IF EXISTS {TABELA_HIVE}")
    spark.sql(f"CREATE TABLE {TABELA_HIVE} USING DELTA LOCATION '{SILVER_PATH}'")

print("Tabela Hive:", TABELA_HIVE)
print("Localização no HDFS:", SILVER_PATH)

Primeira execução: a guardar em Parquet...
Parquet gravado e tabela externa Hive criada.
Tabela Hive: silver.global_superstore_silver
Localização no HDFS: hdfs://hdfs-nn:9000/warehouse/silver.db/global_superstore_silver


In [10]:
# --- Converter o diretório Parquet para Delta ---
if not ja_delta:
    print("A converter Parquet para Delta Lake...")
    spark.sql(f"CONVERT TO DELTA parquet.`{SILVER_PATH}`")

    # Converter a definição no Hive para Delta após converter os ficheiros.
    spark.sql(f"DROP TABLE IF EXISTS {TABELA_HIVE}")
    spark.sql(f"CREATE TABLE {TABELA_HIVE} USING DELTA LOCATION '{SILVER_PATH}'")
    print("Delta Lake pronto e tabela Hive registada como Delta.")
else:
    print("A tabela já era Delta. Conversão dispensada.")

A converter Parquet para Delta Lake...
Delta Lake pronto e tabela Hive registada como Delta.


In [ ]:
# --- 4. VERIFY ---
silver_check = spark.read.format("delta").load(SILVER_PATH)

print("Registos gravados:", silver_check.count())
print("Colunas gravadas:", len(silver_check.columns))
silver_check.printSchema()
silver_check.select("pais", "data_encomenda", "vendas", "desconto", "lucro").show(12, False)

print("\nDescrição da tabela Hive:")
spark.sql(f"DESCRIBE {TABELA_HIVE}").show(30, truncate=False)

print("\nConsulta SQL — Brasil:")
spark.sql(f'''
    SELECT pais,
           COUNT(*) AS linhas_venda,
           ROUND(SUM(vendas), 2) AS total_vendas,
           ROUND(SUM(lucro), 2) AS total_lucro,
           ROUND(100 * SUM(lucro) / NULLIF(SUM(vendas), 0), 2) AS margem_lucro_pct,
           ROUND(100 * AVG(desconto), 2) AS desconto_medio_pct
    FROM {TABELA_HIVE}
    WHERE pais = 'Brazil'
    GROUP BY pais
''').show(truncate=False)

print("\nHistórico Delta:")
spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_PATH}`").show(5, False)

assert silver_check.count() == n_silver, "A gravação não preservou todos os registos."
assert len(silver_check.columns) == 13, "Número de colunas incorreto na Silver."
assert "pais" in silver_check.columns, "Falta a coluna pais."
print("\nSilver Global Superstore criada e verificada com sucesso!")

## Célula 11 — Criar HTML Data Profiling (opcional)

O `YData Profiling` precisa de Pandas. Para evitar consumir muita memória com 51 290 linhas, por defeito usamos uma **amostra de até 10 000 registos**, identificada como tal no título do relatório. Para analisar todos os registos, altera `PERFIL_COMPLETO = True`.

Se faltar a biblioteca: executa `%pip install ydata-profiling` numa célula separada, reinicia o kernel e executa novamente as células necessárias.

In [ ]:
from ydata_profiling import ProfileReport

PERFIL_COMPLETO = False
if PERFIL_COMPLETO:
    perfil_pd = silver_check.toPandas()
    etiqueta = "Completo"
else:
    fracao = min(1.0, 10000 / max(1, silver_check.count()))
    perfil_pd = silver_check.sample(False, fracao, seed=42).limit(10000).toPandas()
    etiqueta = f"Amostra ({len(perfil_pd)} registos)"

profile = ProfileReport(
    perfil_pd,
    title=f"Global Superstore - Data Profiling Silver - {etiqueta}",
    minimal=True
)
profile.to_file(HTML_PATH)
print("Relatório criado:", HTML_PATH)

## Resumo para a apresentação ao professor

| Etapa | Explicação |
|---|---|
| Spark | Configurámos Spark, Hive e Delta Lake. |
| HDFS RAW | Guardámos uma cópia do Excel original e um CSV da folha `Orders`. |
| Bronze | Importámos 51 290 linhas / 24 colunas em Delta, preservando os dados de origem. |
| Extract | Lemos a Bronze e verificámos as colunas essenciais. |
| Transform | Limpámos textos e datas, convertêmo-los para tipos adequados, mantivemos `pais` e criámos quatro flags de qualidade. |
| Validação | Verificámos linhas, colunas, valores em falta e indicadores. |
| Load | Guardámos a Silver em Parquet e criámos uma tabela externa no Hive. |
| Convert | Convertemos Parquet para Delta e registámos a tabela como Delta. |
| Verify | Confirmámos contagens, esquema, histórico Delta e consulta SQL ao Brasil. |
| Profiling | Criámos um relatório HTML, opcionalmente com amostragem. |

**Campos finais:** `data_encomenda`, `ano`, `mercado`, `pais`, `categoria`, `subcategoria`, `vendas`, `desconto`, `lucro`, `flag_vendas_extremas`, `flag_lucro_extremo`, `flag_desconto_alto`, `flag_prejuizo`.

**Nota metodológica:** o World Data 2023 contém uma taxa nacional de desemprego e o Global Superstore contém vendas de vários anos. A ligação por `pais` é feita na Gold e as comparações são exploratórias (não demonstram causalidade).